In [7]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
from src.conexionDB import getEngine

In [8]:
motor = getEngine()

query = """
SELECT
    f.id_fact_contrato,
    f.numero_contrato,
    f.numero_proceso,
    f.valor_contratado,
    f.objeto_del_proceso,
    f.objeto_a_contratar,
    f.origen,

    tf.fecha        AS fecha_firma,
    ti.fecha        AS fecha_inicio,
    tfn.fecha       AS fecha_fin,

    e.codigo_entidad_en_secop,
    e.nombre_entidad,
    e.nit_entidad,
    e.nivel_entidad,
    e.departamento,
    e.municipio,

    p.nombre_proveedor,
    p.tipo_documento,
    p.documento,

    m.modalidad,
    s.estado

FROM analisisfinanciero.fact_contrato f

-- Fecha firma
LEFT JOIN analisisfinanciero.dim_tiempo tf 
    ON f.id_fecha_firma = tf.id_fecha

-- Fecha inicio
LEFT JOIN analisisfinanciero.dim_tiempo ti 
    ON f.id_fecha_inicio = ti.id_fecha

-- Fecha fin
LEFT JOIN analisisfinanciero.dim_tiempo tfn 
    ON f.id_fecha_fin = tfn.id_fecha

-- Dimensiones
LEFT JOIN analisisfinanciero.dim_entidad e 
    ON f.id_entidad = e.id_entidad

LEFT JOIN analisisfinanciero.dim_proveedor p 
    ON f.id_proveedor = p.id_proveedor

LEFT JOIN analisisfinanciero.dim_modalidad m 
    ON f.id_modalidad = m.id_modalidad

LEFT JOIN analisisfinanciero.dim_estado_proceso s 
    ON f.id_estado = s.id_estado;

"""
secop_financiero = pd.read_sql(query, motor)

In [9]:
secop_financiero.shape



(1170546, 21)

Tamaño real del gasto publico

In [12]:
resumen_financiero = {
    'contratos_unicos': secop_financiero.shape[0],
    'valor_total_contratado': secop_financiero['valor_contratado'].sum(),
    'valor_promedio': secop_financiero['valor_contratado'].mean(),
    'valor_mediana': secop_financiero['valor_contratado'].median(),
    'valor_maximo': secop_financiero['valor_contratado'].max(),
    'valor_minimo': secop_financiero['valor_contratado'].min()
}

pd.Series(resumen_financiero)


contratos_unicos          1.170546e+06
valor_total_contratado    1.880170e+14
valor_promedio            1.606233e+08
valor_mediana             1.878000e+07
valor_maximo              4.205028e+12
valor_minimo              0.000000e+00
dtype: float64

In [13]:
secop_financiero['valor_contratado'].quantile(
    [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)


0.25    9.000000e+06
0.50    1.878000e+07
0.75    3.983000e+07
0.90    8.400000e+07
0.95    1.840000e+08
0.99    1.740766e+09
Name: valor_contratado, dtype: float64

Top Entidades por valor 

In [14]:
top_entidades_valor = (
    secop_financiero
    .groupby('nombre_entidad')['valor_contratado']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_entidades_valor


nombre_entidad
DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNOVACION DE MEDELLIN     6.847476e+12
MINISTERIO DE MINAS Y ENERGIA                                        5.846829e+12
RNEC                                                                 4.873450e+12
DEPARTAMENTO DE ANTIOQUIA                                            4.240157e+12
GOBERNACIÓN DE BOYACÁ                                                3.327323e+12
MINISTERIO DE COMERCIO INDUSTRIA Y TURISMO - MINCIT                  3.322074e+12
DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA             3.267138e+12
INSTITUTO DE DESARROLLO URBANO                                       2.594184e+12
ALCALDÍA DEL DISTRITO TURÍSTICO Y CULTURAL DE CARTAGENA DE INDIAS    2.411950e+12
SECRETARÍA DISTRITAL DE INTEGRACIÓN SOCIAL                           2.125532e+12
Name: valor_contratado, dtype: float64

Entidades por numero de contrato 

In [15]:
top_entidades_contratos = (
    secop_financiero['nombre_entidad']
    .value_counts()
    .head(10)
)

top_entidades_contratos


nombre_entidad
INSTITUTO TECNOLOGICO METROPOLITANO                                  17318
SECRETARÍA DISTRITAL DE INTEGRACIÓN SOCIAL                           14508
AGENCIA NACIONAL DE TIERRAS - ANT                                    10565
DEFENSORÍA DEL PUEBLO                                                 8667
DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA              8596
ALCALDÍA DEL DISTRITO TURÍSTICO Y CULTURAL DE CARTAGENA DE INDIAS     8569
SUBRED INTEGRADA DE SERVICIOS DE SALUD SUR OCCIDENTE ESE.             7507
SUBRED INTEGRADA DE SERVICIOS DE SALUD SUR E.S.E.                     7332
SUBRED INTEGRADA DE SERVICIO DE SALUD CENTRO ORIENTE E.S.E 1          7200
SUBRED INTEGRADA DE SERVICIOS DE SALUD NORTE E.S.E. (OFICIAL          6918
Name: count, dtype: int64

Modalidad de contrato


In [16]:
modalidad_dist = (
    secop_financiero['modalidad']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

modalidad_dist


modalidad
CONTRATACIÓN DIRECTA                                                                     64.36
CONTRATACIÓN RÉGIMEN ESPECIAL                                                            12.26
CONTRATACIÓN DIRECTA (LEY 1150 DE 2007)                                                  10.73
MÍNIMA CUANTÍA                                                                            4.02
RÉGIMEN ESPECIAL                                                                          2.70
CONTRATACIÓN MÍNIMA CUANTÍA                                                               1.42
CONTRATACIÓN DIRECTA (CON OFERTAS)                                                        0.85
SELECCIÓN ABREVIADA DE MENOR CUANTÍA                                                      0.83
SELECCIÓN ABREVIADA SUBASTA INVERSA                                                       0.70
CONTRATACIÓN RÉGIMEN ESPECIAL (CON OFERTAS)                                               0.69
CONTRATOS Y CONVENIOS CON MÁS DE DOS PAR

Info hacerca del estado 

In [17]:
estado_dist = (
    secop_financiero['estado']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

estado_dist


estado
MODIFICADO                24.37
EN EJECUCIÓN              24.01
ACTIVO                    16.43
CELEBRADO                 15.38
APROBADO                   9.56
TERMINADO                  5.83
CERRADO                    3.10
CEDIDO                     0.64
SUSPENDIDO                 0.48
LIQUIDADO                  0.14
TERMINADO SIN LIQUIDAR     0.05
CONVOCADO                  0.01
BORRADOR                   0.00
ADJUDICADO                 0.00
EN APROBACIÓN              0.00
NO DEFINIDO                0.00
Name: proportion, dtype: float64

In [18]:
valor_por_estado = (
    secop_financiero
    .groupby('estado')['valor_contratado']
    .sum()
    .sort_values(ascending=False)
)

valor_por_estado


estado
MODIFICADO                7.559863e+13
EN EJECUCIÓN              3.483894e+13
APROBADO                  3.291142e+13
ACTIVO                    2.090519e+13
CELEBRADO                 1.309318e+13
TERMINADO                 6.532700e+12
SUSPENDIDO                2.940167e+12
CERRADO                   7.108958e+11
CEDIDO                    4.473818e+11
LIQUIDADO                 3.015879e+10
CONVOCADO                 3.943395e+09
TERMINADO SIN LIQUIDAR    3.448751e+09
BORRADOR                  6.033300e+08
ADJUDICADO                2.530625e+08
EN APROBACIÓN             9.628310e+07
NO DEFINIDO               6.200000e+06
Name: valor_contratado, dtype: float64

Proveedores influyentes

In [20]:
top_proveedores = (
    secop_financiero
    .groupby('nombre_proveedor')['valor_contratado']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_proveedores


nombre_proveedor
GECELCA S.A. E.S.P.                                                                    4.205028e+12
ZONA FRANCA BARRANQUILLA                                                               2.846224e+12
YESID AVILA TORRES                                                                     2.386780e+12
UNION TEMPORAL INTEGRACION LOGISTICA ELECTORAL 2026                                    2.222283e+12
BANCOLOMBIA                                                                            1.802611e+12
HOSPITAL UNIVERSITARIO SAN JUAN DE DIOS Y MATERNO INFANTIL                             1.615267e+12
IMPRENSA NACIONAL CASA DA MONEDA S.A DE PORTUGAL                                       1.308035e+12
EMPRESA DE DESARROLLO URBANO DE MEDELLIN                                               1.284259e+12
EMPRESA DISTRITAL DE DESARROLLO Y RENOVACIÓN URBANO SOSTENIBLE DE SANTA MARTA- EDUS    1.025937e+12
CONSORCIO K&G SAMUEL MEJIA                                                         

In [25]:
resumen_financiero_final = {
    'contratos_financieros': secop_financiero.shape[0],
    'valor_total_contratado': secop_financiero['valor_contratado'].sum(),
    'entidades_distintas': secop_financiero['nombre_entidad'].nunique(),
    'proveedores_distintos': secop_financiero['nombre_proveedor'].nunique(),
    'modalidades_distintas': secop_financiero['modalidad'].nunique(),
    'duracion_promedio_dias': secop_financiero['duracion_dias'].mean()
}

pd.Series(resumen_financiero_final)

contratos_financieros     1.170546e+06
valor_total_contratado    1.880170e+14
entidades_distintas       6.658000e+03
proveedores_distintos     6.425120e+05
modalidades_distintas     2.600000e+01
duracion_promedio_dias    1.617725e+02
dtype: float64

ANOMALIA: Concentración anómala en proveedores

In [31]:
concentracion_proveedores = (
    secop_financiero
    .groupby('nombre_proveedor')['valor_contratado']
    .sum()
    .sort_values(ascending=False)
)

concentracion_proveedores.head(10)


nombre_proveedor
GECELCA S.A. E.S.P.                                                                    4.205028e+12
ZONA FRANCA BARRANQUILLA                                                               2.846224e+12
YESID AVILA TORRES                                                                     2.386780e+12
UNION TEMPORAL INTEGRACION LOGISTICA ELECTORAL 2026                                    2.222283e+12
BANCOLOMBIA                                                                            1.802611e+12
HOSPITAL UNIVERSITARIO SAN JUAN DE DIOS Y MATERNO INFANTIL                             1.615267e+12
IMPRENSA NACIONAL CASA DA MONEDA S.A DE PORTUGAL                                       1.308035e+12
EMPRESA DE DESARROLLO URBANO DE MEDELLIN                                               1.284259e+12
EMPRESA DISTRITAL DE DESARROLLO Y RENOVACIÓN URBANO SOSTENIBLE DE SANTA MARTA- EDUS    1.025937e+12
CONSORCIO K&G SAMUEL MEJIA                                                         